In [85]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Set seed for benchmarking consistency
np.random.seed(42)

# BENGALURU PINCODE ZONES (30 zones with Demand Weights)
BENGALURU_DATA = {
    "Koramangala": (12.9352, 77.6245, 0.40), "Indiranagar": (12.9719, 77.6412, 0.35),
    "HSR Layout": (12.9121, 77.6446, 0.30), "Whitefield": (12.9698, 77.7500, 0.25),
    "BTM Layout": (12.9166, 77.6101, 0.25), "Jayanagar": (12.9299, 77.5848, 0.20),
    "JP Nagar": (12.9063, 77.5857, 0.20), "MG Road": (12.9738, 77.6119, 0.20),
    "Malleshwaram": (12.9982, 77.5703, 0.15), "Rajajinagar": (12.9901, 77.5525, 0.15),
    "Frazer Town": (12.9975, 77.6144, 0.10), "Hebbal": (13.0354, 77.5988, 0.10),
    "Electronic City": (12.8452, 77.6632, 0.20), "Marathahalli": (12.9569, 77.7011, 0.25),
    "Bellandur": (12.9304, 77.6784, 0.30), "Bannerghatta Rd": (12.8907, 77.5953, 0.15),
    "Ulsoor": (12.9817, 77.6284, 0.15), "Domlur": (12.9610, 77.6387, 0.15),
    "Vasanth Nagar": (12.9880, 77.5885, 0.10), "Richmond Town": (12.9647, 77.5966, 0.10),
    "Kalyan Nagar": (13.0221, 77.6403, 0.10), "Banashankari": (12.9254, 77.5468, 0.10),
    "Basavanagudi": (12.9417, 77.5755, 0.10), "AECS Layout": (12.9615, 77.7135, 0.15),
    "CV Raman Nagar": (12.9793, 77.6642, 0.10), "Cooke Town": (13.0012, 77.6256, 0.05),
    "Yeswanthpur": (13.0235, 77.5550, 0.10), "Kengeri": (12.9038, 77.4833, 0.05),
    "Banaswadi": (13.0104, 77.6482, 0.10), "Sadashiva Nagar": (13.0068, 77.5813, 0.05)
}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlambda = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# --- SIMULATION PARAMETERS ---
sim_start_time = datetime(2026, 4, 14, 19, 0) # 7 PM Peak
duration_min = 180 # 3 Hours
lambda_ph = 1200 # 1200 orders per hour -> ~3600 total orders over 3 hours
total_riders = 1000 # Total unique riders logging in over the 3 hours

# --- 1. GENERATING ORDERS (3 HOURS) ---
zones = list(BENGALURU_DATA.keys())
weights = [BENGALURU_DATA[z][2] for z in zones]
weights = [w/sum(weights) for w in weights]

order_list = []
t_curr = 0
while t_curr < duration_min:
    t_curr += np.random.exponential(60 / lambda_ph)
    if t_curr > duration_min: break

    z_name = np.random.choice(zones, p=weights)
    b_lat, b_lon, _ = BENGALURU_DATA[z_name]

    order_list.append({
        "order_id": len(order_list)+1,
        "zone": z_name,
        "dest_lat": b_lat + np.random.uniform(-0.01, 0.01), # Expanded spread for 3-hr realism
        "dest_lon": b_lon + np.random.uniform(-0.01, 0.01),
        "arrival_time": sim_start_time + timedelta(minutes=t_curr),
        "batch_window": int(t_curr // 3)
    })
orders_df = pd.DataFrame(order_list)

# --- 2. GENERATING DYNAMIC RIDERS ---
rider_list = []
for i in range(total_riders):
    z = np.random.choice(zones)

    # 40% of riders start exactly at 7 PM (T=0)
    # 60% of riders log in randomly between T=0 and T=120 minutes
    if np.random.rand() < 0.40:
        login_offset_min = 0
    else:
        login_offset_min = np.random.uniform(0, 120)

    # Shift length ranges from 1.5 hours to 4 hours
    shift_duration_min = np.random.uniform(90, 240)

    login_time = sim_start_time + timedelta(minutes=login_offset_min)
    logout_time = login_time + timedelta(minutes=shift_duration_min)

    rider_list.append({
        "rider_id": f"R-{i+1:04d}",
        "lat": BENGALURU_DATA[z][0],
        "lon": BENGALURU_DATA[z][1],
        "capacity": np.random.choice([2, 3]),
        "available_at": login_time, # Rider is only available starting from their login time
        "logout_at": logout_time,   # Rider stops taking orders after this time
        "total_dist": 0.0
    })
riders_df = pd.DataFrame(rider_list)

print(f"--- DATA GENERATION COMPLETE ---")
print(f"Total Orders Generated: {len(orders_df)}")
print(f"Total Unique Riders: {len(riders_df)}")
print(f"Time Window: {sim_start_time.strftime('%I:%M %p')} to {(sim_start_time + timedelta(minutes=duration_min)).strftime('%I:%M %p')}")

--- DATA GENERATION COMPLETE ---
Total Orders Generated: 3648
Total Unique Riders: 1000
Time Window: 07:00 PM to 10:00 PM


In [86]:
riders_df

,rider_id,lat,lon,capacity,available_at,logout_at,total_dist
0,R-0001,12.9038,77.4833,2,2026-04-14 20:18:56.323151,2026-04-14 22:20:54.800973,0.0
1,R-0002,13.0221,77.6403,2,2026-04-14 19:22:23.376284,2026-04-14 21:07:45.126743,0.0
2,R-0003,12.9647,77.5966,2,2026-04-14 19:02:03.466709,2026-04-14 21:15:57.205844,0.0
3,R-0004,12.8907,77.5953,2,2026-04-14 19:00:00.000000,2026-04-14 22:54:37.981799,0.0
4,R-0005,12.9615,77.7135,3,2026-04-14 19:01:03.522640,2026-04-14 21:43:54.961055,0.0
...,...,...,...,...,...,...,...
995,R-0996,13.0235,77.5550,3,2026-04-14 19:43:43.940962,2026-04-14 21:35:47.211045,0.0
996,R-0997,12.9166,77.6101,3,2026-04-14 19:00:00.000000,2026-04-14 22:29:22.011015,0.0
997,R-0998,13.0068,77.5813,2,2026-04-14 19:00:00.000000,2026-04-14 22:13:30.181236,0.0
998,R-0999,13.0104,77.6482,2,2026-04-14 19:00:00.000000,2026-04-14 20:54:57.264519,0.0


In [87]:
orders_df

,order_id,zone,dest_lat,dest_lon,arrival_time,batch_window
0,1,Yeswanthpur,13.028140,77.556973,2026-04-14 19:00:01.407804,0
1,2,HSR Layout,12.903262,77.651924,2026-04-14 19:00:01.916679,0
2,3,Bannerghatta Rd,12.881112,77.604698,2026-04-14 19:00:04.673925,0
3,4,Whitefield,12.963436,77.743668,2026-04-14 19:00:10.033214,0
4,5,Hebbal,13.034039,77.594625,2026-04-14 19:00:11.121475,0
...,...,...,...,...,...,...
3643,3644,Banashankari,12.922242,77.543506,2026-04-14 21:59:33.905732,59
3644,3645,Hebbal,13.028214,77.593193,2026-04-14 21:59:46.614939,59
3645,3646,Koramangala,12.934451,77.622879,2026-04-14 21:59:48.587957,59
3646,3647,JP Nagar,12.915951,77.579315,2026-04-14 21:59:53.121303,59


In [88]:
def run_baseline_simulation(orders, riders):
    # Working on copies to preserve original data
    fleet = riders.copy()
    results = []
    avg_speed = 20 # km/h
    service_time = 5 # minutes (pickup + dropoff)

    for _, order in orders.sort_values('arrival_time').iterrows():
        # 1. Identify Pickup Point (Zone Center)
        pickup_lat, pickup_lon, _ = BENGALURU_DATA[order['zone']]

        # 2. Find Nearest Rider (currently available or soonest available)
        # We calculate distance from rider's LAST location to the pickup point
        fleet['dist_to_pickup'] = fleet.apply(
            lambda r: haversine(r['lat'], r['lon'], pickup_lat, pickup_lon), axis=1
        )

        # In baseline, we pick the rider who can reach the restaurant fastest
        best_idx = fleet['dist_to_pickup'].idxmin()
        rider = fleet.loc[best_idx]

        # 3. Calculate Distance Legs
        dist_to_pickup = rider['dist_to_pickup']
        dist_to_dest = haversine(pickup_lat, pickup_lon, order['dest_lat'], order['dest_lon'])
        trip_dist = dist_to_pickup + dist_to_dest

        # 4. Calculate Timing
        travel_time_min = (trip_dist / avg_speed) * 60
        # Rider starts either when order arrives OR when they finish their previous task
        start_time = max(order['arrival_time'], rider['available_at'])
        completion_time = start_time + timedelta(minutes=travel_time_min + service_time)

        # 5. Update Rider State (Fact: They are now at the customer's house)
        fleet.at[best_idx, 'lat'] = order['dest_lat']
        fleet.at[best_idx, 'lon'] = order['dest_lon']
        fleet.at[best_idx, 'available_at'] = completion_time
        fleet.at[best_idx, 'total_dist'] += trip_dist

        # 6. Log Metrics
        # SLA is measured from Arrival Time to Completion Time
        total_duration = (completion_time - order['arrival_time']).total_seconds() / 60
        results.append({
            "order_id": order['order_id'],
            "duration": total_duration,
            "dist": trip_dist,
            "sla_success": total_duration <= 25
        })

    return pd.DataFrame(results), fleet

# EXECUTION
baseline_results, final_fleet = run_baseline_simulation(orders_df, riders_df)

print(f"--- BASELINE METRICS ---")
print(f"Avg Delivery Time: {baseline_results['duration'].mean():.2f} min")
print(f"SLA Success Rate: {baseline_results['sla_success'].mean()*100:.1f}%")
print(f"Total Fleet Distance: {final_fleet['total_dist'].sum():.2f} km")

--- BASELINE METRICS ---
Avg Delivery Time: 34.78 min
SLA Success Rate: 54.7%
Total Fleet Distance: 4932.90 km


In [89]:
def run_batched_simulation(orders, riders):
    fleet = riders.copy()
    results = []
    avg_speed = 20  # km/h
    service_time_per_stop = 5  # minutes (added for each unique dropoff)

    # Sort orders by arrival time
    orders = orders.sort_values('arrival_time').reset_index(drop=True)

    # Tracking which orders are processed
    processed_indices = set()

    for i, order in orders.iterrows():
        if i in processed_indices:
            continue

        # --- BATCHING LOGIC ---
        batch = [order]
        processed_indices.add(i)

        # Look ahead for potential batch mates
        # Constraint 1: Time Window (Arrive within 3 minutes of the first order)
        time_limit = order['arrival_time'] + pd.Timedelta(minutes=3)

        # Check subsequent orders
        for j in range(i + 1, len(orders)):
            potential_order = orders.iloc[j]

            if potential_order['arrival_time'] > time_limit:
                break # Past the time window

            if j in processed_indices:
                continue

            # Constraint 2: Capacity (Max 3 orders)
            if len(batch) >= 3:
                break

            # Constraint 3: Proximity (Within 1.5km of the original order's pickup/zone)
            dist_between_orders = haversine(
                order['dest_lat'], order['dest_lon'],
                potential_order['dest_lat'], potential_order['dest_lon']
            )

            if dist_between_orders <= 1.5:
                batch.append(potential_order)
                processed_indices.add(j)

        # --- DISPATCH LOGIC ---
        # 1. Identify Pickup Point (Assuming same zone for batch for simplicity)
        pickup_lat, pickup_lon, _ = BENGALURU_DATA[order['zone']]

        # 2. Find Nearest Rider
        fleet['dist_to_pickup'] = fleet.apply(
            lambda r: haversine(r['lat'], r['lon'], pickup_lat, pickup_lon), axis=1
        )
        best_idx = fleet['dist_to_pickup'].idxmin()
        rider = fleet.loc[best_idx]

        # 3. Calculate Distance
        # Leg 1: Rider to Restaurant
        dist_to_pickup = rider['dist_to_pickup']

        # Leg 2: Restaurant to all drops (Simplified: Max distance to the furthest drop in batch)
        # In a real batch, this would be a route, but for baseline we use the furthest point
        furthest_drop_dist = max([haversine(pickup_lat, pickup_lon, b['dest_lat'], b['dest_lon']) for b in batch])

        total_trip_dist = dist_to_pickup + furthest_drop_dist

        # 4. Timing
        # Travel time + Service time for EACH order in the batch
        total_travel_time = (total_trip_dist / avg_speed) * 60
        total_service_time = len(batch) * service_time_per_stop

        # Batch starts when the LAST order in the batch arrives or rider is free
        batch_ready_time = max([b['arrival_time'] for b in batch])
        start_time = max(batch_ready_time, rider['available_at'])
        completion_time = start_time + pd.Timedelta(minutes=total_travel_time + total_service_time)

        # 5. Update Rider State (Rider ends at the last drop-off point)
        # We'll use the last order in the batch as the final location
        last_order = batch[-1]
        fleet.at[best_idx, 'lat'] = last_order['dest_lat']
        fleet.at[best_idx, 'lon'] = last_order['dest_lon']
        fleet.at[best_idx, 'available_at'] = completion_time
        fleet.at[best_idx, 'total_dist'] += total_trip_dist

        # 6. Log Metrics for each order in the batch
        for b in batch:
            total_duration = (completion_time - b['arrival_time']).total_seconds() / 60
            results.append({
                "order_id": b['order_id'],
                "duration": total_duration,
                "dist": total_trip_dist / len(batch), # Apportioned distance
                "sla_success": total_duration <= 25
            })

    return pd.DataFrame(results), fleet

# EXECUTION
batch_results, batch_fleet = run_batched_simulation(orders_df, riders_df)

print(f"--- BATCHING METRICS ---")
print(f"Avg Delivery Time: {batch_results['duration'].mean():.2f} min")
print(f"SLA Success Rate: {batch_results['sla_success'].mean()*100:.1f}%")
print(f"Total Fleet Distance: {batch_fleet['total_dist'].sum():.2f} km")

--- BATCHING METRICS ---
Avg Delivery Time: 31.34 min
SLA Success Rate: 65.1%
Total Fleet Distance: 2202.86 km
